# Data Exploration & Preprocessing

### Observations

### Data Preprocessing Steps
- Combine the three datasets into a single `.csv` with the dataset folder in the file path
- create an output `.json` file with Common Name & Scientific Name using ebird code as the key
- Remove any unwanted labels, such as 'spybird' from the secondaries, or anything not in the primaries

In [788]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import json
from tqdm.notebook import tqdm
from collections import Counter
from IPython.display import Audio
import ast

In [789]:
class FilePaths:
    PROJECT_FOLDER = Path('/media/olly/T7/Kaytoo')
    DATA_FOLDER = PROJECT_FOLDER / 'Data'
    DS_TONE = DATA_FOLDER / 'Train_Tier1'
    DS_GRAE = DATA_FOLDER / 'Train_Graeme_Elliot'
    DS_XENO = DATA_FOLDER / 'Train_Xeno_Canto'
    METADATA_TONE = DS_TONE / 'tier1_metadata.csv'
    METADATA_GRAE = DS_GRAE / 'graeme_elliot_metadata.csv'
    METADATA_XENO = DS_XENO / 'xc_metadata.csv'
    OUTPUT_CSV_PATH = DATA_FOLDER / 'Train_Metadata/train_metadata.csv'
    OUTPUT_NAMING_CSV_PATH = DATA_FOLDER / 'Train_Metadata/bird_naming.csv'
    OUTPUT_JSON_PATH = DATA_FOLDER / 'Train_Metadata/kaytoo_bird_names.json'
    BIRD_NAME_MAP = DATA_FOLDER / 'Bird_Names/bird_map_corrected.csv'


In [790]:
class DefaultConfiguration:
    REMOVE_BELOW_THRESHOLD = 5
    SR = 32000
    MERGERS = {'oyster1':['varoys1', 'soioys1', 'blkoys'],
                'parake':['yefpar3', 'refpar4', 'malpar2']}

Now I'm going to extract these configuration settings to variables for the rest of the script.  Doing it this way to make it consistancy with the rest of the project, and so that the analysis can be updated for any changes in dataset settings.

In [791]:
paths = FilePaths()
cfg = DefaultConfiguration()

In [792]:
use_cols = ['filename', 'primary_label', 'secondary_labels', 'start', 'end', 'source_fn']
df_1 = pd.read_csv(paths.METADATA_TONE, usecols=use_cols)
df_1['filename'] = 'Train_Tier1/train_audio/' + df_1['filename'] 
df_2 = pd.read_csv(paths.METADATA_GRAE, usecols=use_cols)
df_2['filename'] = 'Train_Graeme_Elliot/train_audio/' + df_2['filename'] 
df_3 = pd.read_csv(paths.METADATA_XENO, usecols=use_cols)
df_3['filename'] = 'Train_Graeme_Elliot/train_audio/' + df_3['filename'] 
print(f'There are {len(df_1)} rows from Tier 1 data')
df_1.head()

There are 144733 rows from Tier 1 data


,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,yefpar3,['pipipi1'],659,664,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,[],666,671,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'nezfan1']",680,689,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
3,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['gryger1', 'nezbel1', 'tomtit1']",699,704,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
4,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'silver3', 'tomtit1']",709,718,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...


In [793]:
df_2.head()

,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Graeme_Elliot/train_audio/riflem1/00024E...,riflem1,[],3.0,7.0,Paringa 2010/all 30 min counts/{guid {00024E7C...
1,Train_Graeme_Elliot/train_audio/silver3/00024E...,silver3,[],27.0,32.0,Paringa 2010/all 30 min counts/{guid {00024E7C...
2,Train_Graeme_Elliot/train_audio/gryger1/00024E...,gryger1,[],49.0,53.0,Paringa 2010/all 30 min counts/{guid {00024E7C...
3,Train_Graeme_Elliot/train_audio/riflem1/00024E...,riflem1,['tomtit1'],116.0,120.0,Paringa 2010/all 30 min counts/{guid {00024E7C...
4,Train_Graeme_Elliot/train_audio/tomtit1/00024E...,tomtit1,[],147.0,152.0,Paringa 2010/all 30 min counts/{guid {00024E7C...


In [794]:
print(f'There are {len(df_2)} rows in the Graeme Elliot data')

There are 145441 rows in the Graeme Elliot data


In [795]:
df_3.head(3)

,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Graeme_Elliot/train_audio/silver3/silver...,silver3,['spybird'],0.0,12.0,XC895593_Zosterops_lateralis.mp3
1,Train_Graeme_Elliot/train_audio/silver3/silver...,silver3,['spybird'],0.0,12.0,XC841102_Zosterops_lateralis.mp3
2,Train_Graeme_Elliot/train_audio/silver3/silver...,silver3,[],0.0,12.0,XC836020_Zosterops_lateralis.mp3


In [796]:
print(f'There are {len(df_3)} rows in the Xeno Canto data')

There are 2838 rows in the Xeno Canto data


In [797]:
df=pd.concat([df_1, df_2])  #, df_3
df.head()

,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,yefpar3,['pipipi1'],659.0,664.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,[],666.0,671.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'nezfan1']",680.0,689.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
3,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['gryger1', 'nezbel1', 'tomtit1']",699.0,704.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
4,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'silver3', 'tomtit1']",709.0,718.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...


In [798]:
len(df)

290174

## Merge Some Rare Classes

The filepaths will be left the same, we're just going to change the labels.

In [799]:
old_names = all_items = [item for sublist in cfg.MERGERS.values() for item in sublist]
inverted_names = {item: key for key, value_list in cfg.MERGERS.items() for item in value_list}
df['secondary_labels'] = df['secondary_labels'].apply(ast.literal_eval)

mask_1 = (df['primary_label'].isin(old_names))
mask_2 = df['secondary_labels'].apply(lambda x: any(item in old_names for item in x))
len(df[mask_1+ mask_2])

537

In [800]:
df.head(3)

,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,yefpar3,[pipipi1],659.0,664.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,[],666.0,671.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"[nezbel1, nezfan1]",680.0,689.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...


In [801]:
df[df['secondary_labels'].isna()]

,filename,primary_label,secondary_labels,start,end,source_fn


In [802]:
df[mask_2]

,filename,primary_label,secondary_labels,start,end,source_fn
244,Train_Tier1/train_audio/yellow3/AI137_BIRX_120...,yellow3,"[tomtit1, yefpar3]",619.0,624.0,Tier1_ARDs_2011-12/2011-12_5MBC/AI137_BIRX_120...
249,Train_Tier1/train_audio/yellow3/AI137_BIRX_120...,yellow3,"[nezfan1, pipipi1, silver3, yefpar3]",859.0,864.0,Tier1_ARDs_2011-12/2011-12_5MBC/AI137_BIRX_120...
7790,Train_Tier1/train_audio/yellow3/AI137_BIRM_120...,yellow3,"[nezbel1, tomtit1, yefpar3]",103.0,108.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/AI137_BIRM_...
7793,Train_Tier1/train_audio/yellow3/AI137_BIRM_120...,yellow3,"[riflem1, tomtit1, yefpar3]",275.0,280.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/AI137_BIRM_...
8013,Train_Tier1/train_audio/yellow3/AI137_BIRX_120...,yellow3,"[silver3, tomtit1, yefpar3]",105.0,110.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/AI137_BIRX_...
16891,Train_Tier1/train_audio/ausmag2/CS77_BIRD_1111...,ausmag2,"[comcha, eurbla, gryger1, riflem1, yefpar3]",317.0,326.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/CS77_BIRD_1...
16892,Train_Tier1/train_audio/ausmag2/CS77_BIRD_1111...,ausmag2,"[comcha, dunnoc1, eurbla, yefpar3]",336.0,341.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/CS77_BIRD_1...
16893,Train_Tier1/train_audio/ausmag2/CS77_BIRD_1111...,ausmag2,"[comcha, yefpar3]",343.0,348.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/CS77_BIRD_1...
17156,Train_Tier1/train_audio/ausmag2/CS77_BIRX_1111...,ausmag2,[yefpar3],262.0,268.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/CS77_BIRX_1...
17159,Train_Tier1/train_audio/ausmag2/CS77_BIRX_1111...,ausmag2,[yefpar3],312.0,317.0,Tier1_ARDs_2011-12/2011-12_DIURNAL/CS77_BIRX_1...


In [803]:
old_names

['varoys1', 'soioys1', 'blkoys', 'yefpar3', 'refpar4', 'malpar2']

In [804]:
inverted_names['yefpar3']

'parake'

In [805]:
def merge_secondaries(labels, reverse_map=inverted_names, merger_vals = old_names):
    print(labels)
    return [reverse_map[item] if item in merger_vals else item for item in labels]

df.loc[mask_1, 'primary_label'] = df.loc[mask_1, 'primary_label'].map(inverted_names)
df.loc[mask_2, 'secondary_labels'] = df.loc[mask_2, 'secondary_labels'].apply(merge_secondaries)

['tomtit1', 'yefpar3']
['nezfan1', 'pipipi1', 'silver3', 'yefpar3']
['nezbel1', 'tomtit1', 'yefpar3']
['riflem1', 'tomtit1', 'yefpar3']
['silver3', 'tomtit1', 'yefpar3']
['comcha', 'eurbla', 'gryger1', 'riflem1', 'yefpar3']
['comcha', 'dunnoc1', 'eurbla', 'yefpar3']
['comcha', 'yefpar3']
['yefpar3']
['yefpar3']
['eurbla', 'gryger1', 'yefpar3']
['tomtit1', 'yefpar3']
['eurbla', 'maslap1', 'skylar', 'soioys1', 'yellow2']
['eurbla', 'maslap1', 'skylar', 'soioys1']
['comcha', 'eurbla', 'maslap1', 'soioys1', 'yellow2']
['soioys1', 'yellow2']
['eurbla', 'eursta', 'soioys1']
['comcha', 'eurbla', 'soioys1', 'yellow2']


In [806]:
primary_list = sorted(list(df.primary_label.unique()))
primary_list = list(map(str, primary_list) )
primary_list.sort()
print(f'First 5 items of the species list: {primary_list[:5]}')
print(f'Total classes, extracted from the filepaths: {len(primary_list)}')

First 5 items of the species list: ['antpar1', 'ausbit1', 'ausmag2', 'auspip3', 'aussho1']
Total classes, extracted from the filepaths: 106


In [807]:
df.head(3)

,filename,primary_label,secondary_labels,start,end,source_fn
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,parake,[pipipi1],659.0,664.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,[],666.0,671.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"[nezbel1, nezfan1]",680.0,689.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...


## Remove Unwanted Classes

In [808]:
def plot_class_distribution(df, plot_col='primary_label', title='primary_label frequency', height=400):
    total_counts = df[plot_col].value_counts().reset_index()
    total_counts.columns = [plot_col, 'total_count']
    fig = px.bar(total_counts, x=total_counts[plot_col], log_y=True, y=total_counts.total_count, template='seaborn',
    hover_data=[plot_col, 'total_count'], color=plot_col, height=height)
    fig.update_layout(title=title)
    fig.update_layout(showlegend=False)
    fig.update_xaxes(categoryorder='total descending')
    fig.show()

plot_class_distribution(df, plot_col='primary_label', title='Frequency by primary_label')

Let's generate a mapping file for the e-bird to get the scientific and common name in a consistent way

- load up the mapping files from Tier_1
- get a unique row for each ebird code, using the code from the primary_label list
- save the output to csv
- manually check the common names (since there isn't a 1-1 relationship)
- on re-loading, check that all the common names make sense

In [809]:
df_names = pd.read_csv(paths.BIRD_NAME_MAP)
df_names = df_names.drop_duplicates(keep='first', subset='eBird')
df_names = df_names.sort_values(by='eBird').reset_index(drop=True)[['eBird', 'CommonName','ScientificName']]
df_names.head(10)

,eBird,CommonName,ScientificName
0,ausbit1,Australasian Bittern,Botaurus poiciloptilus
1,ausgan1,Australasian Gannet,Morus serrator
2,ausmag2,Australian Magpie,Gymnorhina tibicen
3,auspip3,New Zealand Pipit,Anthus novaeseelandiae
4,aussho1,Australasian Shoveler,Spatula rhynchotis
5,baicra4,Marsh Crake,Zapornia pusilla
6,blafan1,South Island Fantail,Rhipidura fuliginosa
7,blasti1,Black Stilt,Himantopus novaezelandiae
8,blbgul1,Black-billed Gull,Chroicocephalus bulleri
9,blfdot1,Black-fronted Dotterel,Charadrius melanops


In [810]:
len(df_names)

138

In [811]:
df_names.to_csv(paths.OUTPUT_NAMING_CSV_PATH, index=False)
result_list = df_names.to_dict(orient='records')

with paths.OUTPUT_JSON_PATH.open('w') as file:
    json.dump(result_list, file, indent=4)

In [812]:
def map_names(row, mapping):
    common_name = mapping[row['primary_label']][0]
    scientific_name = mapping[row['primary_label']][1]
    return pd.Series([common_name, scientific_name], index=['common_name', 'scientific_name'])

df = df.merge(df_names, left_on='primary_label', right_on='eBird', how='left')
df.drop(columns=['eBird'], inplace=True)
df=df[['filename', 'primary_label', 'CommonName', 'secondary_labels', 'start', 'end', 'source_fn']]
df.head()

,filename,primary_label,CommonName,secondary_labels,start,end,source_fn
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,parake,Parakeet,[pipipi1],659.0,664.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,Brown Creeper,[],666.0,671.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,Brown Creeper,"[nezbel1, nezfan1]",680.0,689.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
3,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,Brown Creeper,"[gryger1, nezbel1, tomtit1]",699.0,704.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...
4,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,Brown Creeper,"[nezbel1, silver3, tomtit1]",709.0,718.0,Tier1_ARDs_2011-12/2011-12_5MBC/AE142_BIRP_120...


In [813]:
df[df['secondary_labels'].isna()].shape

(0, 7)

In [814]:
plot_class_distribution(df, 
                        plot_col='CommonName', 
                        title='Frequency by common_name', 
                        height=800)

In [815]:
total_counts_primary = df['primary_label'].value_counts().reset_index()
total_counts_primary.tail(10)

,primary_label,count
96,yeepen1,1
97,grwpet2,1
98,barpar2,1
99,nezsca1,1
100,nezpig3,1
101,royalb2,1
102,whcalb1,1
103,sopsku1,1
104,stitch1,1
105,categr1,1


Just Checking if we drop the rarest parakeets it won't have a huge impact on the parakeet classes

In [816]:
total_counts_primary[total_counts_primary['primary_label'].isin(['antpar1', 'chipar1', 'malpar2', 'parake', 'refpar4', 'reipar1', 'yefpar3'])]

,primary_label,count
15,parake,2865
94,antpar1,2


So dropping malpar2 and antpar1 will have a negligable difference to the total number of parakeet class.  For now I'm going to just drop anything with less than 5 samples

In [817]:
len(df)

290174

In [818]:
mask = df['primary_label'].map(df['primary_label'].value_counts()) >= cfg.REMOVE_BELOW_THRESHOLD
df = df[mask]
total_counts_common_name = df['CommonName'].value_counts().reset_index()
total_counts_common_name.tail(10)

,CommonName,count
77,Eastern Rosella,11
78,Common Diving-Petrel,9
79,Domestic Chicken,7
80,Spotless Crake,6
81,Chatham Robin,6
82,Spotted Crake,5
83,Chukor,5
84,Black Petrel,5
85,Takahe,5
86,Caspian Tern,5


In [819]:
len(df)

290139

Also drop any with *badbird* or *spybird*

In [820]:
mask = df['primary_label'].isin(['badbird', 'spybird'])
print(len(df[mask]))

8


In [821]:
df=df[~mask]
len(df)

290131

In [822]:
plot_class_distribution(df, 
                        plot_col='CommonName', 
                        title='Frequency by common_name', 
                        height=600)

In [823]:
import ast
primary_list = df['primary_label'].unique()
def filter_list(secondaries, allowed):
    # Convert string representation of list to an actual list
    #actual_list = ast.literal_eval(list_str)
    # Filter the list
    filtered_list = [item for item in secondaries if item in allowed]
    return str(filtered_list)  # Convert back to string if needed

def filter_list(secondaries, allowed):
    if not isinstance(secondaries, (list, str)):
        return secondaries  # Return as is if it's not a list or string
    # Ensure secondaries is a list if it's a string
    if isinstance(secondaries, str):
        secondaries = eval(secondaries)  # Convert string representation of list to actual list
    # Filter the list
    filtered_list = [item for item in secondaries if item in allowed]
    return str(filtered_list)  # Convert back to string if needed




df['secondary_labels'] = df['secondary_labels'].apply(lambda x: filter_list(x, primary_list))

In [824]:
secondary_labels = df['secondary_labels'].to_list()
seconds_list = [eval(string) for string in secondary_labels]
flattened_seconds = [item for sublist in seconds_list for item in sublist]
item_counts = dict(Counter(flattened_seconds))
total_secondaries = len(flattened_seconds)
total_secondaries

110424

It would be good to plot this.

In [825]:
primary_list

array(['parake', 'pipipi1', 'nezfan1', 'gryger1', 'comcha', 'silver3',
       'nezbel1', 'tomtit1', 'eurgre1', 'riflem1', 'yellow3', 'eurbla',
       'nezrob3', 'kea1', 'tui1', 'nezpig2', 'nezrob2', 'shbcuc1',
       'nezkak1', 'eursta', 'sackin1', 'dunnoc1', 'lotkoe1', 'ausmag2',
       'blbgul1', 'kelgul', 'morepo2', 'oyster1', 'sobkiw2', 'mallar3',
       'rebgul1', 'grskiw1', 'nibkiw1', 'bluduc1', 'comred', 'eurgol',
       'chukar', 'nezfal1', 'sonthr1', 'yellow2', 'skylar', 'auspip3',
       'blfter1', 'cangoo', 'calqua', 'baicra4', 'weka1', 'fernbi1',
       'soiwre1', 'parshe1', 'maslap1', 'whfter1', 'whiteh1', 'saddle2',
       'takahe3', 'welswa1', 'chiger2', 'kokako3', 'swahar1', 'litowl1',
       'purswa6', 'commyn', 'houspa', 'charob1', 'x00458', 'piesti1',
       'ausbit1', 'redjun1', 'caster1', 'litpen1', 'liskiw1', 'motpet',
       'blkswa', 'gretea1', 'larus', 'easros1', 'rinphe1', 'compea',
       'dobplo1', 'aussho1', 'spocra2', 'okbkiw1', 'codpet1', 'coopet',
      

In [826]:
len(primary_list)

87

In [827]:
unique_seconds = set(flattened_seconds)
len(unique_seconds)

51

In [828]:
df=df[['filename', 'primary_label', 'secondary_labels']]
df.head()

,filename,primary_label,secondary_labels
0,Train_Tier1/train_audio/yefpar3/AE142_BIRP_120...,parake,['pipipi1']
1,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,[]
2,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'nezfan1']"
3,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['gryger1', 'nezbel1', 'tomtit1']"
4,Train_Tier1/train_audio/pipipi1/AE142_BIRP_120...,pipipi1,"['nezbel1', 'silver3', 'tomtit1']"


In [836]:
list(df['secondary_labels'].unique())

["['pipipi1']",
 '[]',
 "['nezbel1', 'nezfan1']",
 "['gryger1', 'nezbel1', 'tomtit1']",
 "['nezbel1', 'silver3', 'tomtit1']",
 "['gryger1', 'nezbel1']",
 "['silver3']",
 "['gryger1', 'nezfan1', 'silver3']",
 "['gryger1']",
 "['nezbel1', 'tomtit1']",
 "['nezbel1']",
 "['comcha', 'nezbel1']",
 "['nezbel1', 'silver3']",
 "['tomtit1']",
 "['nezbel1', 'nezfan1', 'tomtit1']",
 "['gryger1', 'nezbel1', 'nezfan1']",
 "['nezbel1', 'riflem1']",
 "['riflem1', 'tomtit1']",
 "['gryger1', 'tomtit1']",
 "['comcha', 'gryger1', 'riflem1']",
 "['comcha']",
 "['comcha', 'tomtit1']",
 "['silver3', 'tomtit1']",
 "['tomtit1', 'parake']",
 "['pipipi1', 'riflem1']",
 "['pipipi1', 'riflem1', 'silver3']",
 "['nezfan1', 'pipipi1', 'silver3', 'parake']",
 "['nezfan1', 'pipipi1', 'riflem1', 'tomtit1']",
 "['comcha', 'pipipi1', 'riflem1']",
 "['pipipi1', 'riflem1', 'silver3', 'tomtit1']",
 "['riflem1']",
 "['comcha', 'riflem1']",
 "['riflem1', 'silver3', 'tomtit1']",
 "['nezfan1']",
 "['nezfan1', 'riflem1']",
 "['co

In [830]:
df.to_csv(paths.OUTPUT_CSV_PATH, index=False)

## Varify the audiofiles

I would like to be sure that the audio files can be opend OK

In [831]:
import torchaudio
import torch
import numpy as np


def crop_or_pad(y, length,  train='train', background_paths=None):
    initial_length = len(y)
    if 3 * initial_length < length:
        y = np.concatenate([y,np.zeros(initial_length),y])
    elif 2 * initial_length < length:
        y = np.concatenate([y,np.zeros(initial_length//2),y])
    if len(y) < length:
        y = np.concatenate([y, y]) 
    
    def Normalize(array):
        max_vol = np.abs(array).max()
        return array * 1 / max_vol

    if len(y) < length:
        difference = length - len(y)
        fill=np.zeros(difference)
        y = np.concatenate([y, fill])
    else:
        if train != 'train':
            start = 0
        else:
            start = 0
            start = np.random.randint(len(y) - length)
        y = y[start: start + length]
    y = Normalize(y)
    return y

def open_audio_clip(path, starts=None):    
    try:  
        y, _ = torchaudio.load(path)
        if y.ndim == 2 and y.shape[0] == 2:
            print(f'converting {path} to mono')
            y = torch.mean(y, dim=0).unsqueeze(0)  # from stereo to mono
        y = y.squeeze().numpy() 
    except Exception as e:
        y = np.random.randn(5*320000) 
        print(f'could not open {path}')
        print(e)
    
    if not np.isfinite(y).all():
        y[np.isnan(y)] = np.mean(y)
        y[np.isinf(y)] = np.mean(y)

    y = crop_or_pad(y, 10)  # background_paths=self.back_pths
    return y

In [832]:
tier1_files = [str(paths.DATA_FOLDER / fn) for fn in df_1['filename'].to_list()]
graeme_files = [str(paths.DATA_FOLDER / fn) for fn in df_2['filename'].to_list()]
xeno_files = [str(paths.DATA_FOLDER / fn) for fn in df_3['filename'].to_list()]

tier1_files[:5]

['/media/olly/T7/Kaytoo/Data/Train_Tier1/train_audio/yefpar3/AE142_BIRP_120321_124316_000.flac',
 '/media/olly/T7/Kaytoo/Data/Train_Tier1/train_audio/pipipi1/AE142_BIRP_120321_124316_001.flac',
 '/media/olly/T7/Kaytoo/Data/Train_Tier1/train_audio/pipipi1/AE142_BIRP_120321_124316_002.flac',
 '/media/olly/T7/Kaytoo/Data/Train_Tier1/train_audio/pipipi1/AE142_BIRP_120321_124316_003.flac',
 '/media/olly/T7/Kaytoo/Data/Train_Tier1/train_audio/pipipi1/AE142_BIRP_120321_124316_004.flac']

In [833]:
def play_audio(file_path):
    audio_abe, sr_abe = torchaudio.load(file_path)
    return Audio(data=audio_abe, rate=sr_abe)

play_audio(tier1_files[0])

In [834]:
play_audio(tier1_files[100])

In [835]:
play_audio(graeme_files[1])